# 08 — DistilBERT V2

Amélioration de la V1 avec pondération des classes et seuils ajustés sur validation. Les labels proviennent toujours de GPT-OSS 20B.

In [ ]:
!pip -q install -U transformers datasets accelerate scikit-learn pyarrow


In [ ]:
import time, random, numpy as np, pandas as pd, torch
import torch.nn as nn
from google.colab import files
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_recall_fscore_support, f1_score, accuracy_score, classification_report, confusion_matrix
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:',DEVICE)
if DEVICE=='cuda': print('GPU:',torch.cuda.get_device_name(0))


## Fichiers d’entrée

- `sample_2000.parquet`
- `pairs.parquet`

In [ ]:
uploaded=files.upload()
parquets=[f for f in uploaded if f.lower().endswith('.parquet')]
sample_path=pairs_path=None
for f in parquets:
    d=pd.read_parquet(f); c=set(d.columns)
    if {'segment_id','segment_text','review_id'}.issubset(c) and 'aspect_id' not in c: sample_path=f
    if {'segment_id','aspect_id','sentiment'}.issubset(c): pairs_path=f
assert sample_path and pairs_path
sample=pd.read_parquet(sample_path); pairs=pd.read_parquet(pairs_path)
print('sample',sample.shape,'pairs',pairs.shape)


In [ ]:
ASPECTS=['efficacy_results','hydration_dryness','texture_finish','irritation_sensitivity','acne_breakouts','fragrance_smell','application_absorption','packaging','price_value']
assert sample['segment_id'].is_unique
print(pairs['aspect_id'].value_counts())
print('\nSentiment:')
print(pairs['sentiment'].value_counts())


## 1. Split par `review_id`

In [ ]:
groups=sample['review_id'].astype(str)
g1=GroupShuffleSplit(n_splits=1,test_size=0.15,random_state=SEED)
train_val_idx,test_idx=next(g1.split(sample,groups=groups))
train_val=sample.iloc[train_val_idx].copy(); test_sample=sample.iloc[test_idx].copy()

g2=GroupShuffleSplit(n_splits=1,test_size=0.1765,random_state=SEED+1)
tr_rel,va_rel=next(g2.split(train_val,groups=train_val['review_id'].astype(str)))
train_sample=train_val.iloc[tr_rel].copy(); val_sample=train_val.iloc[va_rel].copy()

print('Train',len(train_sample),'Val',len(val_sample),'Test',len(test_sample))
assert set(train_sample.review_id).isdisjoint(set(val_sample.review_id))
assert set(train_sample.review_id).isdisjoint(set(test_sample.review_id))
assert set(val_sample.review_id).isdisjoint(set(test_sample.review_id))


# A — Détection des aspects

In [ ]:
seg_aspects=pairs.groupby('segment_id')['aspect_id'].apply(lambda s: sorted(set(s.astype(str)))).to_dict()
def make_aspect_df(df):
    out=df[['segment_id','review_id','segment_text']].copy()
    out['aspects']=out['segment_id'].map(seg_aspects).apply(lambda x: x if isinstance(x,list) else [])
    return out

train_aspect=make_aspect_df(train_sample); val_aspect=make_aspect_df(val_sample); test_aspect=make_aspect_df(test_sample)
mlb=MultiLabelBinarizer(classes=ASPECTS); mlb.fit([ASPECTS])
for d in [train_aspect,val_aspect,test_aspect]:
    d['labels']=list(mlb.transform(d['aspects']).astype(np.float32))


In [ ]:
Y=np.stack(train_aspect['labels'].to_numpy())
pos=Y.sum(0); neg=len(Y)-pos
pos_weight=neg/np.clip(pos,1,None)
pos_weight_df=pd.DataFrame({'aspect_id':ASPECTS,'positive_train':pos.astype(int),'negative_train':neg.astype(int),'pos_weight':pos_weight})
display(pos_weight_df)


In [ ]:
MODEL_NAME='distilbert-base-uncased'; MAX_LENGTH=160
aspect_tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
aspect_model=AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,num_labels=len(ASPECTS),problem_type='multi_label_classification',
    id2label={i:a for i,a in enumerate(ASPECTS)},label2id={a:i for i,a in enumerate(ASPECTS)}
)
def tok_a(b): return aspect_tokenizer(b['segment_text'],truncation=True,max_length=MAX_LENGTH)
def to_a(df):
    ds=Dataset.from_pandas(df[['segment_text','labels']].reset_index(drop=True))
    return ds.map(tok_a,batched=True)
tr_a,va_a,te_a=to_a(train_aspect),to_a(val_aspect),to_a(test_aspect)
aspect_collator=DataCollatorWithPadding(aspect_tokenizer)


In [ ]:
class WeightedMultiLabelTrainer(Trainer):
    def __init__(self,*args,pos_weight=None,**kwargs):
        super().__init__(*args,**kwargs); self.pos_weight=pos_weight
    def compute_loss(self,model,inputs,return_outputs=False,num_items_in_batch=None):
        labels=inputs.pop('labels')
        outputs=model(**inputs); logits=outputs.logits
        loss=nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(logits.device))(logits,labels.float())
        return (loss,outputs) if return_outputs else loss

def sigmoid(x): return 1/(1+np.exp(-x))
def aspect_metrics(ep):
    logits,labels=ep
    preds=(sigmoid(logits)>=0.5).astype(int); labels=labels.astype(int)
    pm,rm,fm,_=precision_recall_fscore_support(labels,preds,average='micro',zero_division=0)
    pM,rM,fM,_=precision_recall_fscore_support(labels,preds,average='macro',zero_division=0)
    return {'precision_micro':pm,'recall_micro':rm,'f1_micro':fm,'precision_macro':pM,'recall_macro':rM,'f1_macro':fM}


In [ ]:
aspect_args=TrainingArguments(
    output_dir='/content/aspect_student_v2',learning_rate=2e-5,
    per_device_train_batch_size=16,per_device_eval_batch_size=32,
    num_train_epochs=5,weight_decay=0.01,eval_strategy='epoch',save_strategy='epoch',
    load_best_model_at_end=True,metric_for_best_model='f1_micro',greater_is_better=True,
    logging_steps=20,report_to='none',fp16=torch.cuda.is_available(),seed=SEED
)
aspect_trainer_v2=WeightedMultiLabelTrainer(
    model=aspect_model,args=aspect_args,train_dataset=tr_a,eval_dataset=va_a,
    processing_class=aspect_tokenizer,data_collator=aspect_collator,
    compute_metrics=aspect_metrics,pos_weight=torch.tensor(pos_weight,dtype=torch.float32)
)
aspect_trainer_v2.train()


### Seuils par aspect

Les seuils sont choisis uniquement sur le jeu de validation.

In [ ]:
vp=aspect_trainer_v2.predict(va_a)
vprob=sigmoid(vp.predictions); vtrue=vp.label_ids.astype(int)
candidates=np.arange(0.10,0.91,0.05)
best_thresholds=[]; rows=[]
for i,a in enumerate(ASPECTS):
    best_t,best_f=0.5,-1
    for t in candidates:
        f=f1_score(vtrue[:,i],(vprob[:,i]>=t).astype(int),zero_division=0)
        if f>best_f: best_t,best_f=float(t),float(f)
    best_thresholds.append(best_t)
    rows.append({'aspect_id':a,'best_threshold':best_t,'val_f1':best_f,'val_support_positive':int(vtrue[:,i].sum())})
threshold_df=pd.DataFrame(rows)
display(threshold_df)


In [ ]:
tp=aspect_trainer_v2.predict(te_a)
tprob=sigmoid(tp.predictions); ttrue=tp.label_ids.astype(int)
tbin=(tprob>=np.array(best_thresholds)[None,:]).astype(int)

pm,rm,fm,_=precision_recall_fscore_support(ttrue,tbin,average='micro',zero_division=0)
pM,rM,fM,_=precision_recall_fscore_support(ttrue,tbin,average='macro',zero_division=0)
exact=float(np.mean(np.all(ttrue==tbin,axis=1)))
aspect_v2_summary={'precision_micro':pm,'recall_micro':rm,'f1_micro':fm,'precision_macro':pM,'recall_macro':rM,'f1_macro':fM,'exact_match':exact}
print(aspect_v2_summary)

rows=[]
for i,a in enumerate(ASPECTS):
    p,r,f,_=precision_recall_fscore_support(ttrue[:,i],tbin[:,i],average='binary',zero_division=0)
    rows.append({'aspect_id':a,'threshold':best_thresholds[i],'precision':p,'recall':r,'f1':f,'support_positive':int(ttrue[:,i].sum())})
aspect_v2_per_class=pd.DataFrame(rows).sort_values('f1',ascending=False)
display(aspect_v2_per_class)


# B — Sentiment

In [ ]:
pair_meta=(pairs.drop(columns=['review_id','segment_text'],errors='ignore')
           .merge(sample[['segment_id','review_id','segment_text']].drop_duplicates('segment_id'),
                  on='segment_id',how='left',validate='many_to_one'))
assert pair_meta['review_id'].notna().all() and pair_meta['segment_text'].notna().all()

trr=set(train_sample.review_id.astype(str)); var=set(val_sample.review_id.astype(str)); ter=set(test_sample.review_id.astype(str))
pair_meta['_r']=pair_meta.review_id.astype(str)
train_sent=pair_meta[pair_meta._r.isin(trr)].copy()
val_sent=pair_meta[pair_meta._r.isin(var)].copy()
test_sent=pair_meta[pair_meta._r.isin(ter)].copy()

sent2id={'negative':0,'neutral':1,'positive':2}; id2sent={v:k for k,v in sent2id.items()}
def prep(df):
    out=df.copy()
    out['model_text']='[ASPECT] '+out.aspect_id.astype(str)+' [TEXT] '+out.segment_text.astype(str)
    out['labels']=out.sentiment.map(sent2id)
    assert out['labels'].notna().all()
    return out
train_sent,val_sent,test_sent=prep(train_sent),prep(val_sent),prep(test_sent)
print('Pairs sentiment:',len(train_sent),len(val_sent),len(test_sent))
print(train_sent.sentiment.value_counts())


In [ ]:
counts=train_sent['labels'].value_counts().sort_index().reindex([0,1,2],fill_value=0)
class_weights=len(train_sent)/(3*counts.values)
class_weight_df=pd.DataFrame({'sentiment':['negative','neutral','positive'],'train_count':counts.values,'class_weight':class_weights})
display(class_weight_df)


In [ ]:
sent_tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
sent_model=AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,num_labels=3,id2label=id2sent,label2id=sent2id
)
def tok_s(b): return sent_tokenizer(b['model_text'],truncation=True,max_length=MAX_LENGTH)
def to_s(df):
    ds=Dataset.from_pandas(df[['model_text','labels']].reset_index(drop=True))
    return ds.map(tok_s,batched=True)
tr_s,va_s,te_s=to_s(train_sent),to_s(val_sent),to_s(test_sent)
sent_collator=DataCollatorWithPadding(sent_tokenizer)

class WeightedSentTrainer(Trainer):
    def __init__(self,*args,class_weights=None,**kwargs):
        super().__init__(*args,**kwargs); self.class_weights=class_weights
    def compute_loss(self,model,inputs,return_outputs=False,num_items_in_batch=None):
        labels=inputs.pop('labels')
        outputs=model(**inputs); logits=outputs.logits
        loss=nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))(logits,labels.long())
        return (loss,outputs) if return_outputs else loss

def sent_metrics(ep):
    logits,labels=ep; preds=np.argmax(logits,axis=-1)
    return {'accuracy':accuracy_score(labels,preds),
            'f1_macro':f1_score(labels,preds,average='macro',zero_division=0),
            'f1_weighted':f1_score(labels,preds,average='weighted',zero_division=0)}


In [ ]:
sent_args=TrainingArguments(
    output_dir='/content/sentiment_student_v2',learning_rate=2e-5,
    per_device_train_batch_size=16,per_device_eval_batch_size=32,
    num_train_epochs=5,weight_decay=0.01,eval_strategy='epoch',save_strategy='epoch',
    load_best_model_at_end=True,metric_for_best_model='f1_macro',greater_is_better=True,
    logging_steps=20,report_to='none',fp16=torch.cuda.is_available(),seed=SEED
)
sent_trainer_v2=WeightedSentTrainer(
    model=sent_model,args=sent_args,train_dataset=tr_s,eval_dataset=va_s,
    processing_class=sent_tokenizer,data_collator=sent_collator,compute_metrics=sent_metrics,
    class_weights=torch.tensor(class_weights,dtype=torch.float32)
)
sent_trainer_v2.train()


In [ ]:
sent_v2_metrics=sent_trainer_v2.evaluate(te_s)
print('=== SENTIMENT V2 TEST ===')
print(sent_v2_metrics)
sp=sent_trainer_v2.predict(te_s)
y_true=sp.label_ids; y_pred=np.argmax(sp.predictions,axis=-1)
print(classification_report(y_true,y_pred,target_names=['negative','neutral','positive'],zero_division=0))
cm_v2=confusion_matrix(y_true,y_pred,labels=[0,1,2])
cm_v2_df=pd.DataFrame(cm_v2,index=['gold_negative','gold_neutral','gold_positive'],columns=['pred_negative','pred_neutral','pred_positive'])
display(cm_v2_df)


# C — Comparaison V1 / V2

In [ ]:
v1_aspect={'precision_micro':0.5455,'recall_micro':0.0294,'f1_micro':0.0558,
           'precision_macro':0.0606,'recall_macro':0.0148,'f1_macro':0.0238,'exact_match':0.4267}
comparison_aspect=pd.DataFrame([{'version':'V1',**v1_aspect},{'version':'V2',**aspect_v2_summary}])
display(comparison_aspect)

v1_sent={'accuracy':0.7990,'f1_macro':0.5323,'f1_weighted':0.7898}
comparison_sentiment=pd.DataFrame([
    {'version':'V1',**v1_sent},
    {'version':'V2','accuracy':sent_v2_metrics['eval_accuracy'],
     'f1_macro':sent_v2_metrics['eval_f1_macro'],
     'f1_weighted':sent_v2_metrics['eval_f1_weighted']}
])
display(comparison_sentiment)


# D — Vitesse

In [ ]:
model=aspect_trainer_v2.model.to(DEVICE).eval()
texts=test_aspect.segment_text.tolist()[:200]
enc=aspect_tokenizer(texts,padding=True,truncation=True,max_length=MAX_LENGTH,return_tensors='pt')
enc={k:v.to(DEVICE) for k,v in enc.items()}
if DEVICE=='cuda': torch.cuda.synchronize()
t0=time.perf_counter()
with torch.no_grad(): _=model(**enc)
if DEVICE=='cuda': torch.cuda.synchronize()
elapsed=time.perf_counter()-t0
print('segments:',len(texts))
print('seconds:',round(elapsed,4))
print('ms/segment:',round(1000*elapsed/len(texts),3))
print('segments/s:',round(len(texts)/elapsed,1))


# E — Sauvegarde

In [ ]:
ASPECT_DIR='/content/student_aspect_distilbert_v2'
SENT_DIR='/content/student_sentiment_distilbert_v2'
aspect_trainer_v2.save_model(ASPECT_DIR); aspect_tokenizer.save_pretrained(ASPECT_DIR)
sent_trainer_v2.save_model(SENT_DIR); sent_tokenizer.save_pretrained(SENT_DIR)

threshold_df.to_csv('/content/aspect_thresholds_v2.csv',index=False)
aspect_v2_per_class.to_csv('/content/aspect_metrics_per_class_v2.csv',index=False)
comparison_aspect.to_csv('/content/comparison_aspect_v1_v2.csv',index=False)
comparison_sentiment.to_csv('/content/comparison_sentiment_v1_v2.csv',index=False)
cm_v2_df.to_csv('/content/sentiment_confusion_matrix_v2.csv')
pos_weight_df.to_csv('/content/aspect_pos_weights_v2.csv',index=False)
class_weight_df.to_csv('/content/sentiment_class_weights_v2.csv',index=False)


In [ ]:
!zip -qr /content/sephora_absa_student_v2.zip  /content/student_aspect_distilbert_v2  /content/student_sentiment_distilbert_v2  /content/aspect_thresholds_v2.csv  /content/aspect_metrics_per_class_v2.csv  /content/comparison_aspect_v1_v2.csv  /content/comparison_sentiment_v1_v2.csv  /content/sentiment_confusion_matrix_v2.csv  /content/aspect_pos_weights_v2.csv  /content/sentiment_class_weights_v2.csv

files.download('/content/sephora_absa_student_v2.zip')
